In [ ]:
!pip install ccxt pandas requests matplotlib seaborn scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 4.2 MB/s eta 0:00:00


In [ ]:
import ccxt
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# =====================
# 1. 크라켄 OHLCV 데이터
# =====================
exchange = ccxt.kraken()
symbol = 'BTC/USDT'
timeframe = '1d'  # Fear&Greed는 일별 데이터라 1d로 맞춤
limit = 365

ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
df_price = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df_price['timestamp'] = pd.to_datetime(df_price['timestamp'], unit='ms')
df_price['date'] = df_price['timestamp'].dt.date
df_price = df_price[['date', 'close']].copy()
print("✅ 크라켄 가격 데이터 로드 완료")
print(df_price.tail())

# =====================
# 2. Fear & Greed Index (alternative.me - 무료, 키 불필요)
# =====================
url = "https://api.alternative.me/fng/?limit=365&format=json"
response = requests.get(url)
fng_data = response.json()['data']

df_fng = pd.DataFrame(fng_data)
df_fng['date'] = pd.to_datetime(df_fng['timestamp'], unit='s').dt.date
df_fng['fng_value'] = df_fng['value'].astype(int)
df_fng['fng_class'] = df_fng['value_classification']
df_fng = df_fng[['date', 'fng_value', 'fng_class']].sort_values('date').reset_index(drop=True)
print("\n✅ Fear & Greed 데이터 로드 완료")
print(df_fng.tail())

# =====================
# 3. 데이터 병합
# =====================
df = pd.merge(df_price, df_fng, on='date', how='inner')
df['return_1d'] = df['close'].pct_change() * 100  # 1일 수익률(%)
df = df.dropna().reset_index(drop=True)
print(f"\n✅ 병합 완료 - 총 {len(df)}일 데이터")
print(df.tail())

# =====================
# 4. 상관관계 분석
# =====================
print("\n====== 상관관계 분석 ======")

# 동시 상관관계
corr_same, p_same = stats.pearsonr(df['fng_value'], df['close'])
print(f"FNG ↔ BTC 가격 (동시):     r = {corr_same:.4f}, p = {p_same:.4f}")

corr_ret, p_ret = stats.pearsonr(df['fng_value'], df['return_1d'])
print(f"FNG ↔ BTC 수익률 (동시):   r = {corr_ret:.4f}, p = {p_ret:.4f}")

# =====================
# 5. 선행성 분석 (Cross-correlation)
# =====================
print("\n====== 선행성 분석 (FNG → BTC 수익률) ======")
lags = range(-7, 8)  # -7 ~ +7일 (음수: FNG가 선행, 양수: BTC가 선행)
results = []

for lag in lags:
    if lag < 0:
        # FNG가 lag일 앞서는 경우
        fng_shifted = df['fng_value'].iloc[:lag].values
        ret = df['return_1d'].iloc[-lag:].values
    elif lag > 0:
        fng_shifted = df['fng_value'].iloc[lag:].values
        ret = df['return_1d'].iloc[:-lag].values
    else:
        fng_shifted = df['fng_value'].values
        ret = df['return_1d'].values

    r, p = stats.pearsonr(fng_shifted, ret)
    results.append({'lag': lag, 'correlation': r, 'p_value': p, 'significant': p < 0.05})

df_lag = pd.DataFrame(results)
print(df_lag.to_string(index=False))

best = df_lag.loc[df_lag['correlation'].abs().idxmax()]
print(f"\n📌 최대 상관관계: lag={int(best['lag'])}일, r={best['correlation']:.4f}, p={best['p_value']:.4f}")
if int(best['lag']) < 0:
    print(f"   → FNG가 {abs(int(best['lag']))}일 앞서 BTC 수익률을 선행함")
elif int(best['lag']) > 0:
    print(f"   → BTC 수익률이 {int(best['lag'])}일 앞서 FNG를 선행함")
else:
    print(f"   → 동시 상관관계가 가장 높음")

# =====================
# 6. 시각화
# =====================
fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle('BTC Fear & Greed Index vs Price Analysis', fontsize=16, fontweight='bold')

# 그래프 1: 가격 + FNG 추이
ax1 = axes[0]
ax2 = ax1.twinx()
ax1.plot(df['date'], df['close'], color='orange', label='BTC Price', linewidth=1.5)
ax2.fill_between(df['date'], df['fng_value'], alpha=0.3, color='blue', label='FNG')
ax2.plot(df['date'], df['fng_value'], color='blue', linewidth=0.8)
ax1.set_ylabel('BTC Price (USDT)', color='orange')
ax2.set_ylabel('Fear & Greed Index', color='blue')
ax1.set_title('BTC Price vs Fear & Greed Index')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylim(0, 100)

# 그래프 2: 산점도 (FNG vs 수익률)
axes[1].scatter(df['fng_value'], df['return_1d'], alpha=0.4, color='purple', s=15)
z = np.polyfit(df['fng_value'], df['return_1d'], 1)
p_line = np.poly1d(z)
x_line = np.linspace(df['fng_value'].min(), df['fng_value'].max(), 100)
axes[1].plot(x_line, p_line(x_line), "r--", linewidth=2, label=f'추세선 (r={corr_ret:.3f})')
axes[1].axhline(y=0, color='gray', linestyle='-', alpha=0.3)
axes[1].axvline(x=50, color='gray', linestyle='-', alpha=0.3)
axes[1].set_xlabel('Fear & Greed Index')
axes[1].set_ylabel('BTC 1D Return (%)')
axes[1].set_title('FNG vs BTC 일별 수익률 산점도')
axes[1].legend()

# 그래프 3: Lag별 상관관계
colors = ['red' if s else 'steelblue' for s in df_lag['significant']]
axes[2].bar(df_lag['lag'], df_lag['correlation'], color=colors, alpha=0.7)
axes[2].axhline(y=0, color='black', linewidth=0.8)
axes[2].set_xlabel('Lag (일) — 음수: FNG 선행, 양수: BTC 선행')
axes[2].set_ylabel('Pearson r')
axes[2].set_title('Cross-Correlation: FNG → BTC 수익률 (빨간색 = p<0.05)')
axes[2].set_xticks(list(lags))

plt.tight_layout()
plt.savefig('fng_btc_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ 그래프 저장 완료: fng_btc_analysis.png")